# Merval Financial Analysis — Exploratory Data Analysis

In this notebook we connect to the Alpha Vantage API and explore 
historical price data for the top Merval stocks.

## Objectives
- Connect to Alpha Vantage API
- Extract historical price data for 5 Merval stocks
- Understand the structure of the data
- Identify missing values or anomalies

In [ ]:
import requests
import pandas as pd 
import os
from dotenv import load_dotenv

# Cargar API key desde .env
load_dotenv()
API_KEY = os.getenv("ALPHA_VANTAGE_KEY")

# Acciones a analizar
tickers = {
    "YPF": "YPF",
    "Galicia": "GGAL",
    "MercadoLibre": "MELI",
    "Tenaris": "TS",
    "Pampa": "PAM"
}

# Test - traer datos de YPF
url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=YPF&outputsize=compact&apikey={API_KEY}"
response = requests.get(url)
data = response.json()

print("Keys en la respuesta:", list(data.keys()))

Keys en la respuesta: ['Meta Data', 'Time Series (Daily)']


In [2]:
# Ver metadata
print ("Meta Data:")
print(data['Meta Data'])

# Ver primeras 3 fechas
time_series = data['Time Series (Daily)']
fechas = list(time_series.keys())[:3]
print(f"\nÚltimas 3 fechas disponibles: {fechas}")
print(f"\nEstructura de un día:")
print(time_series[fechas[0]])

Meta Data:
{'1. Information': 'Daily Prices (open, high, low, close) and Volumes', '2. Symbol': 'YPF', '3. Last Refreshed': '2026-03-24', '4. Output Size': 'Compact', '5. Time Zone': 'US/Eastern'}

Últimas 3 fechas disponibles: ['2026-03-24', '2026-03-23', '2026-03-20']

Estructura de un día:
{'1. open': '41.5100', '2. high': '43.4700', '3. low': '41.4900', '4. close': '42.5600', '5. volume': '1825381'}


In [4]:
import time

def get_stock_data(symbol, api_key):
    url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&outputsize=compact&apikey={api_key}"
    response = requests.get(url)
    data = response.json()
    
    if 'Time Series (Daily)' not in data:
        print(f"Error en {symbol}: {data}")
        return None
    
    time_series = data['Time Series (Daily)']
    df = pd.DataFrame(time_series).T
    df.index = pd.to_datetime(df.index)
    df.columns = ['open', 'high', 'low', 'close', 'volume']
    df = df.astype(float)
    df['symbol'] = symbol
    return df

# Traer datos de todas las acciones
dfs = []
for nombre, ticker in tickers.items():
    print(f"Descargando {nombre} ({ticker})...")
    df = get_stock_data(ticker, API_KEY)
    if df is not None:
        dfs.append(df)
    time.sleep(15)  # Esperar 15 segundos entre llamadas

df_all = pd.concat(dfs)
df_all = df_all.reset_index().rename(columns={'index': 'date'})

print(f"\nDimensiones: {df_all.shape}")
print(f"\nAcciones: {df_all['symbol'].unique()}")
print(f"\nRango de fechas: {df_all['date'].min()} → {df_all['date'].max()}")

Descargando YPF (YPF)...
Descargando Galicia (GGAL)...
Descargando MercadoLibre (MELI)...
Descargando Tenaris (TS)...
Descargando Pampa (PAM)...

Dimensiones: (500, 7)

Acciones: <StringArray>
['YPF', 'GGAL', 'MELI', 'TS', 'PAM']
Length: 5, dtype: str

Rango de fechas: 2025-10-28 00:00:00 → 2026-03-24 00:00:00


In [5]:
# Vista general
print(df_all.dtypes)
print(f"\nNulos por columna:")
print(df_all.isnull().sum())
print(f"\nMuestra de datos:")
print(df_all.head())

date      datetime64[us]
open             float64
high             float64
low              float64
close            float64
volume           float64
symbol               str
dtype: object

Nulos por columna:
date      0
open      0
high      0
low       0
close     0
volume    0
symbol    0
dtype: int64

Muestra de datos:
        date   open   high    low  close     volume symbol
0 2026-03-24  41.51  43.47  41.49  42.56  1825381.0    YPF
1 2026-03-23  41.00  42.42  41.00  41.21  3458347.0    YPF
2 2026-03-20  41.62  43.40  41.03  41.92  5984290.0    YPF
3 2026-03-19  39.75  42.60  39.70  41.59  7904585.0    YPF
4 2026-03-18  37.79  39.79  37.68  39.48  2151981.0    YPF


In [6]:
# Resumen por acción
resumen = df_all.groupby('symbol').agg(
    ultimo_cierre=('close', 'last'),
    precio_max=('high', 'max'),
    precio_min=('low', 'min'),
    volumen_promedio=('volume', 'mean'),
    dias_disponibles=('date', 'count')
).round(2)

print(resumen)

        ultimo_cierre  precio_max  precio_min  volumen_promedio  \
symbol                                                            
GGAL            55.05       62.52       40.54        1422627.91   
MELI          2295.92     2428.00     1606.21         575134.44   
PAM             82.46       94.50       69.34         223472.70   
TS              38.08       57.95       37.69        1499120.65   
YPF             33.85       43.47       32.32        1851343.13   

        dias_disponibles  
symbol                    
GGAL                 100  
MELI                 100  
PAM                  100  
TS                   100  
YPF                  100  
